## Node-style hooks函数用法
支持两种用法
- 装饰器是函数式挂载，把一个hook快速挂载到Agent的某个节点。
- 类写法是对象化中间件，把中间件封装为一个可配置、可复用、可扩展的组件。

In [3]:
# 基于装饰器实现
# 1、模型的初始化
import os
from dotenv import load_dotenv
from langchain_qwq import ChatQwen

custom_profile = {
"max_input_tokens": 128_000 # 最大上下文长度
}

# 从.env文件中加载环境变量
load_dotenv(override=True)
# 模型的初始化
model = ChatQwen(
    model="qwen3.6-flash",
    api_base=os.getenv("DASHSCOPE_API_BASE"),  # 国内 Key 必须用国内地址
    profile=custom_profile, # 手动添加的配置项
)

In [ ]:
from langchain.agents.middleware import (
    before_model,
    after_model,
    before_agent,
    after_agent,
    AgentState,
    AgentMiddleware,
)
from langchain.messages import HumanMessage
from langgraph.runtime import Runtime
from langchain.agents import create_agent
from typing import Any


# 1. 定义 before_model 钩子
@before_model
def before_model_middleware(
    state: AgentState,  # 当前运行的状态，包含了用户的消息列表等信息
    runtime: Runtime # 是这次运行时的上下文和基础设施。
) -> dict[str, Any] | None:
    state["messages"][-1].content += " -> before_model <- "
    return None


# 2. 定义 after_model 钩子
@after_model
def after_model_middleware(
    state: AgentState, runtime: Runtime
) -> dict[str, Any] | None:
    state["messages"][-1].content += " -> after_model <- "
    return None


# 3. 定义 before_agent 钩子
@before_agent
def before_agent_middleware(
    state: AgentState, runtime: Runtime
) -> dict[str, Any] | None:
    state["messages"][-1].content += " -> before_agent <- "
    return None


# 4. 定义 after_agent 钩子
@after_agent
def after_agent_middleware(state: AgentState, runtime: Runtime) -> None:
    state["messages"][-1].content += " -> after_agent <- "
    return None


agent = create_agent(
    model=model,
    middleware=[
        before_model_middleware,
        after_model_middleware,
        before_agent_middleware,
        after_agent_middleware,
    ],  # 👈 添加中间件
)

response = agent.invoke({
    "messages": [HumanMessage("你好啊")],
})

for msg in response["messages"]:

    msg.pretty_print()

================================ Human Message =================================

你好啊 -> before_agent <-  -> before_model <- 
================================== Ai Message ==================================

你好！已收到你的输入流标识：`你好啊 -> before_agent <-  -> before_model <-`。

在日常交互中，这通常表示数据会先经过 `before_agent`（如权限校验、路由、日志记录等），再进入 `before_model`（如 Prompt 组装、上下文裁剪、安全过滤等），最后才由大模型生成回复。

如果你是在调试自定义 Hook、中间件或工作流框架，我可以：
- 帮你梳理各阶段的输入/输出格式
- 提供对应代码片段（Python/JS/Node.js 等）
- 模拟后续 `model_output -> after_model <-` 的完整流程

需要我侧重哪一部分？随时告诉我～ 😊 -> after_model <-  -> after_agent <-


# 基于类实现
1. 必须继承 AgentMiddleware ← 这个固定
2. 方法名固定 ( before_model , after_model ) ← 这个固定
3. 类名随意 ← 这个不固定

LangGraph 只看：

是否继承 AgentMiddleware？

是否有 before_model / after_model 等方法？

In [3]:
from langchain.agents.middleware import AgentMiddleware, AgentState, hook_config
from langchain.messages import HumanMessage
from langgraph.runtime import Runtime
from langchain.agents import create_agent
from typing import Any


class MyMiddleware(AgentMiddleware):
    def __init__(self):
        super().__init__()

    def before_model(
        self, state: AgentState, runtime: Runtime
    ) -> dict[str, Any] | None:
        print("before_model")
        print(state)
        print(runtime)
        state["messages"][-1].content += " -> before_model <- "
        return None

    def after_model(
        self, state: AgentState, runtime: Runtime
    ) -> dict[str, Any] | None:
        state["messages"][-1].content += " -> after_model <- "
        return None

    def before_agent(
        self, state: AgentState, runtime: Runtime
    ) -> dict[str, Any] | None:
        state["messages"][-1].content += " -> before_agent <- "
        return None

    def after_agent(self, state: AgentState, runtime: Runtime) -> None:
        state["messages"][-1].content += " -> after_agent <- "
        return None


my_middleware = MyMiddleware()

agent = create_agent(
    model=model,
    middleware=[my_middleware],
)

response = agent.invoke({
    "messages": [HumanMessage("你好啊")],
})

for msg in response["messages"]:
    msg.pretty_print()

before_model
{'messages': [HumanMessage(content='你好啊 -> before_agent <- ', additional_kwargs={}, response_metadata={}, id='110a3421-4254-463b-9c13-338257781aa3')]}
Runtime(context=None, store=None, stream_writer=<function Pregel.stream.<locals>.stream_writer at 0x118afeac0>, heartbeat=<function _no_op_heartbeat at 0x108c8a5c0>, previous=None, execution_info=ExecutionInfo(checkpoint_id='1f1794fd-9a61-6cd4-8001-8afa611864cd', checkpoint_ns='MyMiddleware.before_model:b7b86670-38da-59a3-584b-4284c08d3fc4', task_id='b7b86670-38da-59a3-584b-4284c08d3fc4', thread_id=None, run_id=None, node_attempt=1, node_first_attempt_time=1783351928.611798), server_info=None, control=<langgraph.runtime.RunControl object at 0x10ff373a0>)
================================ Human Message =================================

你好啊 -> before_agent <-  -> before_model <- 
================================== Ai Message ==================================

你好呀！👋  
你输入的格式里带了一些类似框架运行时的标记（`before_agent` / `before_model`），不过这不

1. before_model通用场景：
- 消息修剪（trim messages）
- PII 脱敏
- 输入验证
- 条件路由

2.  after_model 通常的场景：
- 输出验证
- 格式化响应
- 统计信息
- 状态更新

### 2种方法的统一
装饰器底层会基于我们重写的方法构造一个AgentMiddleware子类的实例，以@after_model为例：

In [ ]:
# 这是after_model最终返回的内容
return type(
    middleware_name,
    (AgentMiddleware,),
    {
        "state_schema": state_schema or AgentState,
        "tools": tools or [],
        "after_model": wrapped,
},
)()

这是after_model最终返回的内容。

上述代码中的wrapped是after_model内部的装饰器，代码如下


In [ ]:
def wrapped(
    _self: AgentMiddleware[StateT, ContextT],
    state: StateT,
    runtime: Runtime[ContextT],
) -> dict[str, Any] | Command[Any] | None:
    return func(state, runtime) # type: ignore[return-value]

In [ ]:
return type(
    middleware_name,
    (AgentMiddleware,),
    {
        "state_schema": state_schema or AgentState,
        "tools": tools or [],
        "after_model": func(state, runtime),
    },
)()

而 func(state, runtime) 正是我们定义的、被 @after_model 修饰的函数，在上述案例中对应的是
after_model_middleware，上述代码的含义是

    1. 创建一个AgentMiddleware的子类
   
    2. 类名为middleware_name，即创建agent时传递的中间件名称，上述案例中after_model_middleware

    3. 这个子类有两个属性 state_schema 和 tools

    4. 有一个方法： after_model ，逻辑等同于 func(state, runtime) 。

    5. 最后的括号 () 表示实例化子类，返回一个对象

#### 参数说明
- state: 是一个AgentState实例，维护Agent运行中的状态，这类状态会随着Agent的运行而发生变化，包括消息列表。

- runtime: 是一个Runtime实例，维护Agent运行过程中的上下文环境，包括上下文、长期记忆等。

#### 返回值说明
- 返回None:不修改状态

- 返回字典：更新状态

- 返回{“jump_to”:...}：控制流程
```python
def before_model(self, state, runtime):
    if state.get("count", 0) > 10:
        return {"jump_to": "__end__"} # 跳过模型，直接结束
    return None
```
  - "__end__" - 结束 Agent
  - "tools" - 跳到工具节点
  - 其他自定义节点

#### 装饰器参数：can_jump_to
涉及到Node-style的四个hook函数可以接收额外参数 `can_jump_to` 。
钩子函数可以 `改变Agent正常的运行轨迹` 。比如：发现上下文窗口溢出，直接跳转至结尾，提前终止整
个Agent。
can_jump_to 决定了钩子函数可以直接跳转至流程的哪些位置，可取值如下：
- end：跳转至Agent流程末尾，或第一个after_agent钩子，直接终止整个流程。
- tools：跳转至工具节点。
- model：跳转至模型节点，或第一个before_model钩子。

In [6]:
from typing import Any

from langchain.agents import create_agent
from langchain.agents.middleware import before_model, after_model,before_agent,after_agent, AgentState
from langchain.messages import AIMessage, SystemMessage
from langchain.tools import tool
from langgraph.runtime import Runtime
from rich import print as rprint


@tool
def get_news() -> str:
    """获取当日新闻"""
    return "美加墨世界杯今日开幕"


# 在模型（LLM）执行前触发。允许跳转到 "tools" 节点。
@before_model(can_jump_to=["tools"])
def force_tool_first(
    state: AgentState, runtime: Runtime
) -> dict[str, Any] | None:
    """
    【业务场景：强行拦截并触发工具】
    如果用户输入包含 "direct tool"，则跳过本次大模型的思考/生成阶段，
    直接伪造一个大模型的 tool_calls 意图，强行把控制权移交给工具执行节点。
    """
    text = state["messages"][-1].content

    # 检查关键词，满足条件则强行干预流程
    if isinstance(text, str) and "direct tool" in text.lower():
        print("[MIDDLEWARE] before_model: jump_to='tools'")

        # 人工构造一个大模型的消息对象（AIMessage）
        # 欺骗系统，让系统误以为这是模型自己决定要调用的工具
        fake_tool_call = AIMessage(
            content="人工构造的消息",
            tool_calls=[
                {
                    "name": "get_news",
                    "args": {},
                    "id": "call_force_weather_001",
                }
            ],
        )

        # 返回更新后的状态：注入伪造的消息，并明确指定下一步跳转到 "tools" 节点
        return {
            "messages": [fake_tool_call],
            "jump_to": "tools",
        }

    # 如果不满足触发条件，返回 None，流程正常向下流转（继续让 LLM 思考）
    return None


# 在模型（LLM）执行生成之后触发。允许重新跳转回 "model" 节点。
@after_model(can_jump_to=["model"])
def retry_with_extra_instruction(
    state: AgentState, runtime: Runtime
) -> dict[str, Any] | None:
    """
    【业务场景：反思/重试机制】
    如果大模型已经生成了回答，但发现用户最初的请求包含 "retry model"，
    则动态追加一条系统提示词（SystemMessage），强行让模型重新生成（重试）一次。
    """
    # 倒序遍历消息历史，找到最近的一次用户输入（human 消息）
    user_text = ""
    for msg in reversed(state["messages"]):
        if getattr(msg, "type", "") == "human":
            user_text = getattr(msg, "content", "")
            break

    # 检查用户输入是否包含触发重试的关键字
    if isinstance(user_text, str) and "retry model" in user_text.lower():
        # 【核心防御】：防止无限循环重跳（死循环）
        # 检查消息历史中是否已经注入过这条特殊的系统提示。如果有，说明已经重试过了，不再重复干预。
        already_injected = any(
            isinstance(getattr(msg, "content", None), str)
            and "你必须以【二次回答】开头" in msg.content
            for msg in state["messages"]
        )
        if already_injected:
            return None  # 已注入过，直接放行，结束重试流程

        print(
            "[MIDDLEWARE] after_model: jump_to='model' with extra system instruction"
        )

        # 返回更新后的状态：追加强力约束的系统消息，并将指针跳回 "model" 节点重新执行
        return {
            "messages": [
                SystemMessage("你必须以【二次回答】开头，并且只用一句话回答。")
            ],
            "jump_to": "model",
        }

    return None


# 在模型（LLM）执行前触发。允许直接跳转到 "end" 节点（强行终止）。
@before_agent(can_jump_to=["end"])
def overflow_context_processor(
    state: AgentState, runtime: Runtime
) -> dict[str, Any] | None:
    """
    【业务场景：安全卫士/异常拦截】
    模拟上下文窗口溢出（Token超限）或其他严重的系统阻断情况。
    一旦触发，直接熔断流程，拒绝让大模型继续处理，直接报错或返回兜底文案。
    """
    # 假装溢出,模拟检查最后一条消息是否包含 overflow 标识
    if "overflow" in state["messages"][-1].content:
        print(
            "[MIDDLEWARE] before_model: jump_to='end' when contenxt window overflow"
        )

        # 构造兜底的结束消息，并直接指定跳转到 "end" 终止 Agent 运行
        return {
            "messages": [AIMessage("上下文窗口溢出，终止")],
            "jump_to": "end",
        }

    return None


agent = create_agent(
    model=model,
    tools=[get_news],
    # 将定义的中间件按照顺序挂载到 Agent 中（注意：执行顺序会严格按照列表声明顺序）
    middleware=[
        force_tool_first,
        retry_with_extra_instruction,
        overflow_context_processor,
    ],
)


def run_once(user_input: str):
    result = agent.invoke({
        "messages": [{"role": "user", "content": user_input}],
    })
    rprint(result)

if __name__ == "__main__":
    # Case 1: 直接跳 tools
    # 预期表现：
    # 1. 触发 force_tool_first，打印 "[MIDDLEWARE] before_model: jump_to='tools'"
    # 2. 绕过 LLM 的首轮思考，直接调用 `get_news` 工具
    # 3. 工具返回结果后，LLM 总结工具结果并输出
    print("=" * 30, "-> Case 1 <-", "=" * 30)
    run_once("请帮我查今日新闻 direct tool overflow")

    # Case 2: 输出后跳回 model
    # 预期表现：
    # 1. 正常进入 LLM 生成第 1 版回答
    # 2. 触发 retry_with_extra_instruction，打印 "[MIDDLEWARE] after_model: jump_to='model'..."
    # 3. 注入系统提示词后，LLM 被强行拉回并生成第 2 版回答
    # 4. 最终输出应带有“【二次回答】”前缀
    print("=" * 30, "-> Case 2 <-", "=" * 30)
    run_once("请随便介绍一下 LangChain retry model")

    # Case 3:
    # 预期表现：
    # 1. 触发 overflow_context_processor 中间件
    # 2. 直接打印终止信息并退出，LLM 根本不会接收到这个请求
    print("=" * 30, "-> Case 3 <-", "=" * 30)
    run_once("你好 overflow")

    # Case 4: 正常流程
    # 预期表现：
    # 1. 没有任何中间件被触发（不满足任何关键字）
    # 2. Agent 走正常的 OOTB（Out of the box）标准工作流：User -> Model -> Call Tool -> Model -> End
    print("=" * 30, "-> Case 4 <-", "=" * 30)
    run_once("今日新闻摘要？")

============================== -> Case 1 <- ==============================
[MIDDLEWARE] before_model: jump_to='end' when contenxt window overflow


{
    'messages': [
        HumanMessage(
            content='请帮我查今日新闻 direct tool overflow',
            additional_kwargs={},
            response_metadata={},
            id='981487f8-db42-4334-90c7-945bf3a3d323'
        ),
        AIMessage(
            content='上下文窗口溢出，终止',
            additional_kwargs={},
            response_metadata={},
            id='eb008807-c16a-4940-bdc4-96f0b7c23149',
            tool_calls=[],
            invalid_tool_calls=[]
        )
    ]
}

============================== -> Case 2 <- ==============================
[MIDDLEWARE] after_model: jump_to='model' with extra system instruction


{
    'messages': [
        HumanMessage(
            content='请随便介绍一下 LangChain retry model',
            additional_kwargs={},
            response_metadata={},
            id='1abc834c-e7d0-438a-a3bd-7a602626c2ef'
        ),
        AIMessage(
            content='LangChain 中的 **重试机制（Retry Mechanism）** 是一个面向 LLM API 
调用的“容错基础设施”，主要用于应对网络抖动、服务端临时过载、速率限制（Rate 
Limit）或偶发超时等问题。它的核心思想是：**当一次请求失败时，不立刻报错，而是按预设策略自动重试，直到成功或达到上限
。**\n\n下面用比较直白的方式给你拆解一下它是怎么工作的、怎么用以及需要注意什么：\n\n---\n\n### 🔍 
为什么需要重试？\n大模型 API 大多是远程服务，调用链路长且不可控。典型故障场景：\n- 并发太高触发 `429 Too Many 
Requests`\n- 数据中心节点切换导致 `503 Service Unavailable`\n- 网络瞬断或 DNS 解析慢引发 `Timeout`\n- 某些 Provider
返回短暂的服务降级\n\n如果不加重试，这些**瞬时失败**会直接打穿你的应用；加上重试后，90% 
以上的抖动都能被自动消化。\n\n---\n\n### ⚙️ 底层原理与实现方式\nLangChain 本身不硬编码重试逻辑，而是由各 Provider 
客户端（如 `langchain_openai`, `langchain_anthropic` 等）基于成熟的重试库（通常是 `tenacity` 
或自封装）实现。主流做法是：\n- **指数退避（Exponential Backoff）**：第一次失败等 1 秒，第二次 2 秒，第三次 4 秒… 
避免频繁请求雪崩式压垮服务端。\n- **可配置异常过滤**：只针对特定错误码/异常类型重试，跳过客户端参数错误（如非法 
token 格式）。\n- **HTTP 层面拦截**：重试发生在发起请求 → 接收响应 → 解析结果 之前，对业务代码透明。\n\n---\n\n### 
💡 基础用法示例\n以 OpenAI Provider 为例（最新 `langchain-openai` 包）：\n```python\nfrom langchain_openai import 
ChatOpenAI\n\nllm = ChatOpenAI(\n    model="gpt-4o-mini",\n    max_retries=3,      # 最多重试 3 次（含初始调用共 4 
次尝试）\n    timeout=30,         # 单次请求超时 30 秒\n    temperature=0.7\n)\n\nresponse = 
llm.invoke("请用一句话解释量子计算")\nprint(response.content)\n```\n不同 Provider 
的参数名可能略有差异，但基本都包含 
`max_retries`、`timeout`，部分支持自定义退避策略。你可以通过源码或文档查看具体支持的配置项。\n\n---\n\n### 🧩 
进阶技巧与搭配方案\n| 场景 | 推荐做法 |\n|------|----------|\n| **重试仍失败怎么办？** | 使用 
`.with_fallbacks([backup_llm])` 优雅降级到备用模型或旧版本 |\n| **想记录重试过程？** | 配合 LangChain Callbacks 
监听 `on_retry` 事件，输出日志或指标 |\n| **非幂等操作（如批量写入）？** | ❌ 
谨慎开启重试！可能导致重复执行。建议关闭或仅重试网络层错误 |\n| **严格延迟要求的服务？** | 调小 `max_retries` 和 
`timeout`，甚至关闭重试，改用熔断/快速失败策略 |\n\n---\n\n### ⚠️ 常见误区\n1. **重试 ≠ 
解决根本问题**：如果是账号余额不足、Key 无效、模型下架，重试只会徒增延迟。\n2. **盲目提高重试次数**：超过 5 
次往往边际收益极低，反而拖慢整体吞吐量。\n3. **忽略 Provider 官方 SDK 的限制**：部分厂商自带限流保护，LangChain 
的重试会被其底层再次拦截。\n\n---\n\n### 📦 总结一句话\n> LangChain 的重试机制是一个 
**“开箱即用、参数可调、指数退避”** 的 API 
容错开关，适合处理瞬时故障，但不替代完善的监控、降级与限流体系。合理配置后，能让你的 LLM 
应用从“一抖就挂”变成“稳如老狗”。\n\n如果你想看某个具体 Provider（如 
Anthropic、Ollama、阿里云百炼）的重试配置写法，或者想结合 `RetryPolicy` 写自定义逻辑，我可以继续展开。',
            additional_kwargs={
                'refusal': None,
                'reasoning_content': 'Here\'s a thinking process:\n\n1.  **Analyze User Input:**\n   - **Topic:** 
LangChain retry model (LangChain 重试模型机制)\n   - **Request:** "请随便介绍一下" (Please introduce it 
casually/randomly)\n   - **Language:** Chinese\n\n2.  **Identify Key Concepts:**\n   - What is LangChain? A 
framework for building applications with LLMs.\n   - What is the "retry" mechanism in LangChain? It\'s part of 
LangChain\'s resilience/error-handling features, specifically related to interacting with LLM providers (API 
calls).\n   - Key components: `RetryPolicy`, `with_fallbacks`, exponential backoff, error handling strategies.\n   
- How it works: When an API call fails (rate limit, timeout, server error), LangChain can automatically retry with 
configurable parameters.\n   - Implementation: Typically done via `ChatOpenAI(..., max_retries=N)` or higher-level 
abstractions like `RetryPolicy` or middleware/callbacks.\n\n3.  **Structure the Response (in Chinese):**\n   - 
Introduction/Overview: What is it and why it\'s needed.\n   - Core Mechanism: How retries work in LangChain.\n   - 
Key Features/Configuration: Parameters like max retries, wait time, exceptions to catch.\n   - Usage Examples: Code
snippet showing basic usage.\n   - Advanced/Related Concepts: `with_fallbacks`, custom retry logic, integration 
with other tools.\n   - Best Practices/Tips: When to use, pitfalls to avoid.\n   - Conclusion: Summary.\n\n4.  
**Draft Content (Mental Refinement in Chinese):**\n   *(Introduction)* LangChain 的重试机制（Retry 
Mechanism）主要是为了解决大语言模型（LLM）API 调用中常见的临时性问题，比如网络波动、速率限制（Rate 
Limit）、服务器超时或偶发错误。自动重试可以显著提升应用的稳定性和用户体验。\n   \n   *(Core Mechanism)*

============================== -> Case 3 <- ==============================
[MIDDLEWARE] before_model: jump_to='end' when contenxt window overflow


{
    'messages': [
        HumanMessage(
            content='你好 overflow',
            additional_kwargs={},
            response_metadata={},
            id='0fa63d36-6f20-4c0a-b9d6-631e891a4a07'
        ),
        AIMessage(
            content='上下文窗口溢出，终止',
            additional_kwargs={},
            response_metadata={},
            id='c9eb81c6-cd66-4654-9283-5758fa375962',
            tool_calls=[],
            invalid_tool_calls=[]
        )
    ]
}

============================== -> Case 4 <- ==============================


{
    'messages': [
        HumanMessage(
            content='今日新闻摘要？',
            additional_kwargs={},
            response_metadata={},
            id='29887649-7b88-45d0-b248-500b7740d9ec'
        ),
        AIMessage(
            content='',
            additional_kwargs={
                'refusal': None,
                'reasoning_content': 'Here\'s a thinking process:\n\n1.  **Analyze User Input:**\n   - User asks: 
"今日新闻摘要？" (Today\'s news summary?)\n   - This is a direct request for current news summaries.\n\n2.  
**Identify Available Tools:**\n   - I have one tool available: `get_news` with the description "获取当日新闻" (Get 
today\'s news).\n   - The tool takes no parameters.\n\n3.  **Determine Action:**\n   - Since the user is asking for
today\'s news, I should call the `get_news` tool to fetch the data.\n\n4.  **Execute Tool Call:**\n   - Call 
`get_news()` with empty parameters as specified.\n\n5.  **Prepare Response (Mental):**\n   - After receiving the 
tool output, I will format it into a concise summary for the user.\n   - Since I haven\'t actually called it yet, 
I\'ll proceed with the tool call first.\n\n   *Self-Correction/Verification during thought:*\n   - The tool schema 
says `parameters: {properties: {}, type: "object"}`, so calling it with `{}` or no arguments is correct.\n   - I 
will generate the tool call now.✅'
            },
            response_metadata={
                'token_usage': {
                    'completion_tokens': 279,
                    'prompt_tokens': 254,
                    'total_tokens': 533,
                    'completion_tokens_details': {
                        'accepted_prediction_tokens': None,
                        'audio_tokens': None,
                        'reasoning_tokens': 261,
                        'rejected_prediction_tokens': None,
                        'text_tokens': 279
                    },
                    'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': None, 'text_tokens': 254}
                },
                'model_provider': 'dashscope',
                'model_name': 'qwen3.6-flash',
                'system_fingerprint': None,
                'id': 'chatcmpl-17c8b108-56b4-96ba-bea7-c89630b85eb2',
                'finish_reason': 'tool_calls',
                'logprobs': None
            },
            id='lc_run--019f387c-3de5-7e00-a802-1bd97e9d995a-0',
            tool_calls=[
                {'name': 'get_news', 'args': {}, 'id': 'call_c43e247a1fe6483dba239f79', 'type': 'tool_call'}
            ],
            invalid_tool_calls=[],
            usage_metadata={
                'input_tokens': 254,
                'output_tokens': 279,
                'total_tokens': 533,
                'input_token_details': {},
                'output_token_details': {'reasoning': 261}
            }
        ),
        ToolMessage(
            content='美加墨世界杯今日开幕',
            name='get_news',
            id='165394a4-8582-4e16-ac1d-558e94ca45a9',
            tool_call_id='call_c43e247a1fe6483dba239f79'
        ),
        AIMessage(
            content='美加墨世界杯今日开幕',
            additional_kwargs={
                'refusal': None,
                'reasoning_content': '用户询问今日新闻摘要，我已经调用了 `get_news` 
工具并获取到了相关信息（美加墨世界杯今日开幕）。现在我将基于这个结果生成回复。'
            },
            response_metadata={
                'token_usage': {
                    'completion_tokens': 46,
                    'prompt_tokens': 291,
                    'total_tokens': 337,
                    'completion_tokens_details': {
                        'accepted_prediction_tokens': None,
                        'audio_tokens': None,
                        'reasoning_tokens': 35,
                        'rejected_prediction_tokens': None,
                        'text_tokens': 46
                    },
                    'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': None, 'text_tokens': 291}
                },
                'model_provider': 'dashscope'

1. 我们提前判定需要调用工具，直接在before_model中跳转至工具节点，省去了一次模型调用
2. 通过约定的 retry model标记 ，在after_model之后再次跳转到模型节点，触发模型重复调用
3. 通过约定的 overflow标记 ，模拟上下文窗口溢出，在before_model中直接跳转至结尾，提前终止流程
4. Case 4 是没有被干预的正常Agent流程，作为对照。

##### 基于类的实现
和基于装饰器实现的关键区别在于：需要引入额外的装饰器 `@hook_config` 为 `can_jump_to` 传参

In [ ]:
from typing import Any

from langchain.agents import create_agent
from langchain.agents.middleware import hook_config, AgentState, AgentMiddleware
from langchain.messages import AIMessage, SystemMessage
from langchain.tools import tool
from langgraph.runtime import Runtime


@tool
def get_news() -> str:
    """获取当日新闻"""
    return "美加墨世界杯今日开幕"


class MyMiddleware(AgentMiddleware):
    @hook_config(can_jump_to=["tools", "end"])
    def before_model(
        self, state: AgentState, runtime: Runtime
    ) -> dict[str, Any] | None:
        text = state["messages"][-1].content

        # 假装溢出
        if "overflow" in text:
            print(
                "[MIDDLEWARE] before_model: jump_to='end' when contenxt window overflow"
            )
            return {
                "messages": [AIMessage("上下文窗口溢出，终止")],
                "jump_to": "end",
            }

        if isinstance(text, str) and "direct tool" in text.lower():
            print("[MIDDLEWARE] before_model: jump_to='tools'")
            fake_tool_call = AIMessage(
                content="人工构造的消息",
                tool_calls=[
                    {
                        "name": "get_news",
                        "args": {},
                        "id": "call_force_weather_001",
                    }
                ],
            )
            return {
                "messages": [fake_tool_call],
                "jump_to": "tools",
            }

        return None

    @hook_config(can_jump_to=["model"])
    def after_model(
        self, state: AgentState, runtime: Runtime
    ) -> dict[str, Any] | None:
        user_text = ""
        for msg in reversed(state["messages"]):
            if getattr(msg, "type", "") == "human":
                user_text = getattr(msg, "content", "")
                break

        if isinstance(user_text, str) and "retry model" in user_text.lower():
            # 防止无限重跳：如果已经加过提示，就不再跳
            already_injected = any(
                isinstance(getattr(msg, "content", None), str)
                and "你必须以【二次回答】开头" in msg.content
                for msg in state["messages"]
            )
            if already_injected:
                return None

            print(
                "[MIDDLEWARE] after_model: jump_to='model' with extra system instruction"
            )
            return {
                "messages": [
                    SystemMessage("你必须以【二次回答】开头，并且只用一句话回答。")
                ],
                "jump_to": "model",
            }

        return None


agent = create_agent(
    model=model,
    tools=[get_news],
    middleware=[MyMiddleware()],
)


def run_once(user_input: str):
    result = agent.invoke({
        "messages": [{"role": "user", "content": user_input}],
    })
    for msg in result["messages"]:
        msg.pretty_print()


if __name__ == "__main__":
    # Case 1: 直接跳 tools
    print("=" * 30, "-> Case 1 <-", "=" * 30)
    run_once("请帮我查今日新闻 direct tool")

    # Case 2: 输出后跳回 model
    print("=" * 30, "-> Case 2 <-", "=" * 30)
    run_once("请随便介绍一下 LangChain retry model")

    # Case 3:
    print("=" * 30, "-> Case 3 <-", "=" * 30)
    run_once("你好 overflow")

    # Case 4: 正常流程
    print("=" * 30, "-> Case 4 <-", "=" * 30)
    run_once("今日新闻摘要？")